In [10]:
class Model(nn.Module):
    def __init__(self, N: int, k: float, r: float):
        super().__init__()
        self.N = N
        
        target_params = int((N ** k) * r)
        
        # Ricalcoliamo H sapendo che l'ultimo strato ora mappa H -> N
        # P = (N * H + H) + (H * N + N) = 2*H*N + H + N
        # H = (P - N) // (2*N + 1)
        H = max(1, (target_params - N) // (2 * N + 1))
        
        self.net = nn.Sequential(
            nn.Linear(N, H),
            nn.ReLU(),
            nn.Linear(H, N), # Output di dimensione N per combaciare con i sample
            nn.Sigmoid()     # Opzionale: vincola l'output in [0, 1] come i bit reali
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Prende (batch_size, N) e sputa fuori (batch_size, N)
        return self.net(x)

    def get_actual_parameter_count(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [5]:
def gaussian_kernel(x: torch.Tensor, y: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """
    Calcola la matrice del kernel Gaussiano tra x e y.
    x: shape (v_size, N)
    y: shape (w_size, N)
    """
    x_size = x.size(0)
    y_size = y.size(0)
    dim = x.size(1)

    # Espansione delle dimensioni per calcolare le distanze pairwise
    x_expanded = x.unsqueeze(1).expand(x_size, y_size, dim)
    y_expanded = y.unsqueeze(0).expand(x_size, y_size, dim)

    # Distanza euclidea al quadrato: ||x_i - y_j||²
    distances = torch.pow(x_expanded - y_expanded, 2).sum(2)
    
    # Calcolo del kernel RBF
    return torch.exp(-distances / (2 * sigma ** 2))

def mmd_squared(x: torch.Tensor, y: torch.Tensor, sigma: float = 1.0) -> torch.Tensor:
    """
    Calcola la MMD² empirica tra due set di campioni X e Y.
    X: Campioni reali (batch del training set), shape (batch_size, N)
    Y: Campioni generati/predetti dal modello, shape (batch_size, N)
    """
    k_xx = gaussian_kernel(x, x, sigma)
    k_yy = gaussian_kernel(y, y, sigma)
    k_xy = gaussian_kernel(x, y, sigma)

    # Formula empirica della MMD²
    loss = k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()
    return loss

In [12]:
N = 10
model = Model(N=N, k=2, r=0.5)

print(f"Parametri Target: {int((N**2)*0.5)}")
print(f"Parametri Effettivi: {model.get_actual_parameter_count()}")

# Shape: (batch_size, N)
my_batch = torch.randint(0, 2, (64, N)).float()

# Forward pass del modello per ottenere i campioni approssimati
campioni_modello = model(my_batch)

# Calcolo della loss diretta
loss = mmd_squared(my_batch, campioni_modello, sigma=0.1)

loss.backward()

Parametri Target: 50
Parametri Effettivi: 31


In [17]:
import torch.optim as optim

# Iperparametri
N = 10
k = 2
r = 0.5
epochs = 1000
batch_size = 32
lr = 0.005
sigma_mmd = 0.1

# Istanza del modello e dell'ottimizzatore (eseguiamo su CPU)
model = Model(N=N, k=k, r=r)
optimizer = optim.Adam(model.parameters(), lr=lr)

# Generiamo un dataset fittizio (sostituisci questo con i tuoi veri sample del circuito)
# Struttura: (Numero_totale_di_sample, N)
num_samples = 500
X_train = torch.randint(0, 2, (num_samples, N)).float()

print(f"Parametri effettivi del modello: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
print("Inizio del training...")

# ==========================================
# 3. TRAINING LOOP
# ==========================================

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    # Permutazione casuale dei dati ad ogni epoca per creare i batch
    permutation = torch.randperm(X_train.size(0))
    num_batches = 0
    
    for i in range(0, X_train.size(0), batch_size):
        # 1. Estrazione del batch dal training set
        indices = permutation[i:i + batch_size]
        batch_real = X_train[indices]
        
        # Saltiamo i batch incompleti (es. di dimensione 1) per evitare instabilità nella MMD
        if batch_real.size(0) < 2:
            continue
            
        # 2. Reset dei gradienti
        optimizer.zero_grad()
        
        # 3. Forward pass: il modello genera i campioni approssimati
        batch_generated = model(batch_real)
        
        # 4. Calcolo della loss MMD² empirica
        loss = mmd_squared(batch_real, batch_generated, sigma=sigma_mmd)
        
        # 5. Backward pass e aggiornamento dei pesi
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
    # Stampa del progresso ogni 10 epoche
    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / num_batches
        print(f"Epoca [{epoch+1}/{epochs}] - MMD² Loss Media: {avg_loss:.6f}")

print("Training completato!")

Parametri effettivi del modello: 31
Inizio del training...
Epoca [1/1000] - MMD² Loss Media: 0.597918
Epoca [10/1000] - MMD² Loss Media: 0.216314
Epoca [20/1000] - MMD² Loss Media: 0.173452
Epoca [30/1000] - MMD² Loss Media: 0.156070
Epoca [40/1000] - MMD² Loss Media: 0.151013
Epoca [50/1000] - MMD² Loss Media: 0.147009
Epoca [60/1000] - MMD² Loss Media: 0.145429
Epoca [70/1000] - MMD² Loss Media: 0.133746
Epoca [80/1000] - MMD² Loss Media: 0.132985
Epoca [90/1000] - MMD² Loss Media: 0.128441
Epoca [100/1000] - MMD² Loss Media: 0.133015
Epoca [110/1000] - MMD² Loss Media: 0.131875
Epoca [120/1000] - MMD² Loss Media: 0.123950
Epoca [130/1000] - MMD² Loss Media: 0.123399
Epoca [140/1000] - MMD² Loss Media: 0.126744
Epoca [150/1000] - MMD² Loss Media: 0.119461
Epoca [160/1000] - MMD² Loss Media: 0.120055
Epoca [170/1000] - MMD² Loss Media: 0.120754
Epoca [180/1000] - MMD² Loss Media: 0.113979
Epoca [190/1000] - MMD² Loss Media: 0.119217
Epoca [200/1000] - MMD² Loss Media: 0.114078
Epoca [